# MCTS Pipeline
Mirrors `IMPLEMENTATION_CHECKLIST.md` Part 2. Each section tests the corresponding `src/` code. Run cells in order.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
if (_cwd / 'src').exists():
    REPO_ROOT = _cwd
elif (_cwd.parent / 'src').exists():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot find src/ from {_cwd}. Launch Jupyter from the repo root or notebooks/.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.fm_model import FMInference, MODEL_PATH
from src.matchup_db import MatchupDB, DEFAULT_SAVE_PATH
from src.draft_state import DraftState, available_actions, apply_pick

fm = FMInference.load()
db = MatchupDB.load()

# vocab is the authoritative brawler list — must match what the FM was trained on
VOCAB = tuple(fm.schema.vocab)

print(f"FM loaded  : D={len(fm.w_linear)} features, k={fm.V.shape[1]}, {len(VOCAB)} brawlers")
print(f"MatchupDB  : {len(db.brawler['global'])} brawlers in global stats")
print()

# skill_tier_boundaries is None if matchup_db.pkl was built before Step 2.1.
# Rebuild with: db = MatchupDB.build(); db.save()
# Not needed until Step 2.4 (rollout policy uses matchup DB tier lookups).
if db.skill_tier_boundaries is None:
    print("NOTE: db.skill_tier_boundaries is None — matchup_db.pkl needs a rebuild before Step 2.4.")
    print("      (No impact on Steps 2.1–2.3.)")
else:
    q25, q50, q75 = db.skill_tier_boundaries
    print(f"Skill tier boundaries (Q25/Q50/Q75): {q25:.3f} / {q50:.3f} / {q75:.3f}")

---
## 2.1 — Draft State Representation
**Source:** `src/draft_state.py`

Two things to verify:
1. `DraftState` fields are correct and `whose_turn` matches the confirmed pick order at every position for both P1 and P2.
2. `available_actions()` returns exactly the right brawlers (vocab minus picked minus banned), and `apply_pick()` transitions state correctly.

In [ ]:
# ── 2.1.1: State fields + whose_turn verification ─────────────────────────────
# Use real brawlers from the vocabulary.
MY_PICK   = VOCAB[0]   # e.g. '8-BIT'
OPP_PICK  = VOCAB[1]   # e.g. 'ALLI'

mid_draft = DraftState(
    my_team=frozenset({MY_PICK}),
    opp_team=frozenset({OPP_PICK}),
    mode="gemGrab",
    map_name="Double Swoosh",
    skill_ns=1.2,
    is_first_pick=True,
)

print("── Mid-draft state (P1, after pick 0 and pick 1) ──")
print(f"  my_team      : {mid_draft.my_team}")
print(f"  opp_team     : {mid_draft.opp_team}")
print(f"  mode         : {mid_draft.mode}")
print(f"  map_name     : {mid_draft.map_name}")
print(f"  skill_ns     : {mid_draft.skill_ns}")
print(f"  is_first_pick: {mid_draft.is_first_pick}")
print(f"  bans         : {mid_draft.bans}  (empty — no bans entered)")
print(f"  pick_number  : {mid_draft.pick_number}  (0-based; 2 picks made)")
print(f"  whose_turn   : '{mid_draft.whose_turn}'  (expect 'opp' — P1 sequence slot 2)")
print(f"  is_terminal  : {mid_draft.is_terminal}  (expect False)")
print()

# Walk through the full P1 and P2 sequences and assert every whose_turn is correct.
P1_ORDER = ("mine", "opp", "opp", "mine", "mine", "opp")
P2_ORDER = ("opp",  "mine", "mine", "opp",  "opp",  "mine")

print("── Pick-order walk-through ──────────────────────────")
print(f"{'Pick':>4}  {'P1 expected':>12}  {'P1 actual':>10}  {'P2 expected':>12}  {'P2 actual':>10}  {'OK?':>4}")
print("-" * 62)

picks = list(VOCAB[:6])
s_p1 = DraftState(my_team=frozenset(), opp_team=frozenset(),
                  mode="gemGrab", map_name="Double Swoosh",
                  skill_ns=1.2, is_first_pick=True)
s_p2 = DraftState(my_team=frozenset(), opp_team=frozenset(),
                  mode="gemGrab", map_name="Double Swoosh",
                  skill_ns=1.2, is_first_pick=False)

for i, (exp_p1, exp_p2) in enumerate(zip(P1_ORDER, P2_ORDER)):
    act_p1 = s_p1.whose_turn
    act_p2 = s_p2.whose_turn
    ok = "✓" if act_p1 == exp_p1 and act_p2 == exp_p2 else "✗"
    print(f"{i:>4}  {exp_p1:>12}  {act_p1:>10}  {exp_p2:>12}  {act_p2:>10}  {ok:>4}")
    s_p1 = apply_pick(s_p1, picks[i])
    s_p2 = apply_pick(s_p2, picks[i])

assert s_p1.is_terminal and s_p2.is_terminal, "Both drafts should be terminal after 6 picks"
print(f"\nis_terminal after 6 picks: P1={s_p1.is_terminal}  P2={s_p2.is_terminal}  ✓")

In [ ]:
# ── 2.1.2: available_actions and apply_pick ────────────────────────────────────

# 1. Count after 2 picks — expect 93 from 95-brawler vocab
state_2picks = DraftState(
    my_team=frozenset({VOCAB[0]}),
    opp_team=frozenset({VOCAB[1]}),
    mode="gemGrab", map_name="Double Swoosh",
    skill_ns=1.2, is_first_pick=True,
)
actions = available_actions(state_2picks, VOCAB)
assert len(actions) == len(VOCAB) - 2, f"Expected {len(VOCAB)-2}, got {len(actions)}"
assert VOCAB[0] not in actions and VOCAB[1] not in actions, "Picked brawler appeared in actions"
print(f"After 2 picks: {len(actions)} available  (expect {len(VOCAB)-2})  ✓")
print(f"  Picked '{VOCAB[0]}' in actions: {VOCAB[0] in actions}  (expect False)")
print(f"  Picked '{VOCAB[1]}' in actions: {VOCAB[1] in actions}  (expect False)")

# 2. Bans restrict the pool
BANS = frozenset({VOCAB[2], VOCAB[3], VOCAB[4]})
state_banned = DraftState(
    my_team=frozenset({VOCAB[0]}),
    opp_team=frozenset({VOCAB[1]}),
    mode="gemGrab", map_name="Double Swoosh",
    skill_ns=1.2, is_first_pick=True,
    bans=BANS,
)
actions_banned = available_actions(state_banned, VOCAB)
expected = len(VOCAB) - 2 - len(BANS)
assert len(actions_banned) == expected, f"Expected {expected}, got {len(actions_banned)}"
assert not (BANS & set(actions_banned)), "Banned brawler appeared in actions"
print(f"\nWith {len(BANS)} bans: {len(actions_banned)} available  (expect {expected})  ✓")
print(f"  Banned brawlers in actions: {BANS & set(actions_banned)}  (expect empty set)")

# 3. apply_pick: walk through one full draft and verify team membership
print("\n── apply_pick: full draft walkthrough ──────────────")
state = DraftState(
    my_team=frozenset(), opp_team=frozenset(),
    mode="gemGrab", map_name="Double Swoosh",
    skill_ns=1.2, is_first_pick=True,
)
# Use the first 6 available brawlers in vocab order
draft_picks = list(VOCAB[:6])
print(f"{'Pick':>4}  {'By':>6}  {'Brawler':<14}  my_team                opp_team")
print("-" * 72)
for b in draft_picks:
    turn = state.whose_turn
    state = apply_pick(state, b)
    print(f"{state.pick_number:>4}  {turn:>6}  {b:<14}  "
          f"{str(set(state.my_team)):<23}  {set(state.opp_team)}")

assert state.is_terminal
assert len(state.my_team) == 3 and len(state.opp_team) == 3
print(f"\nFinal state — my_team: {state.my_team}  |  opp_team: {state.opp_team}")
print(f"is_terminal: {state.is_terminal}  ✓")

---
## 2.2 — MCTS Tree Structure
**Source:** `src/mcts_node.py`

Two things to verify:
1. `UCB1_C = 0.5` and `ucb1_score()` select the expected child on a hand-crafted tree — both at low C (exploit) and high C (explore).
2. `select()` traverses from a root with expanded children down to the first unexpanded leaf without errors.

In [ ]:
# ── 2.2.1 + 2.2.2: UCB1 selects the expected child ───────────────────────────
# Hand-craft root → 3 children with known visit counts and values.
# No FM needed — pure tree arithmetic.
from src.mcts_node import MCTSNode, UCB1_C, ucb1_score, select_child

_state = DraftState(
    my_team=frozenset(), opp_team=frozenset(),
    mode="gemGrab", map_name="Double Swoosh",
    skill_ns=1.0, is_first_pick=True,
)

root = MCTSNode(state=_state, visit_count=100)
# A: high Q (0.80), moderate visits — dominates at low C
# B: low Q  (0.50), few visits     — gets exploration bonus at high C
# C: mid Q  (0.55), many visits
root.children = {
    "A": MCTSNode(state=_state, parent=root, visit_count=40, value_sum=32.0),
    "B": MCTSNode(state=_state, parent=root, visit_count=10, value_sum=5.0),
    "C": MCTSNode(state=_state, parent=root, visit_count=50, value_sum=27.5),
}

print(f"Default UCB1_C = {UCB1_C}\n")
print(f"{'Child':>6}  {'Q':>6}  {'UCB1 (C=0.1)':>14}  {'UCB1 (C=2.0)':>14}")
print("-" * 46)
for name, child in root.children.items():
    s_low  = ucb1_score(child, root.visit_count, c=0.1)
    s_high = ucb1_score(child, root.visit_count, c=2.0)
    print(f"{name:>6}  {child.q_value:>6.3f}  {s_low:>14.4f}  {s_high:>14.4f}")

best_low,  _ = select_child(root, c=0.1)
best_high, _ = select_child(root, c=2.0)
print(f"\nC=0.1 → selects '{best_low}'  (expect A — highest Q wins at low C)")
print(f"C=2.0 → selects '{best_high}'  (expect B — fewest visits wins at high C)")

assert best_low  == "A", f"Expected A, got {best_low}"
assert best_high == "B", f"Expected B, got {best_high}"
print("\n✓ UCB1 selection correct at both extremes")

In [ ]:
# ── 2.2.2: select() traverses root → leaf ─────────────────────────────────────
# Expand one level with real brawlers so select() has visited children to pass
# through, landing at the first unexpanded grandchild.
from src.mcts_node import select

root2 = MCTSNode(state=_state, visit_count=50)
root2.expand(VOCAB)

# Give every root child some visits so UCB1 is meaningful (no +inf ties)
for i, child in enumerate(root2.children.values()):
    child.visit_count = i + 1
    child.value_sum   = (i + 1) * 0.55   # Q ≈ 0.55 for all, slight variation

path = select(root2)

print(f"Path length : {len(path)}  (expect 2 — root + one unexpanded grandchild)")
print(f"Path[0] is root : {path[0] is root2}  ✓")
print(f"Path[-1].is_leaf: {path[-1].is_leaf}  ✓  (grandchild not yet expanded)")
print(f"Path[-1].state.pick_number: {path[-1].state.pick_number}  (expect 1)")

assert path[0] is root2
assert path[-1].is_leaf
assert path[-1].state.pick_number == 1
assert len(path) == 2
print("\n✓ select() reaches a leaf in one step from an expanded root")

---
## 2.3 — FM Integration
**Source:** `src/fm_integration.py`

Two things to verify:
1. `evaluate()` returns a probability in (0, 1) for a terminal state, raises `ValueError` for a partial state, and satisfies the team-symmetry constraint: P(A beats B) + P(B beats A) ≈ 1.0.
2. The result cache works: a second call to the same state is a cache hit and returns the identical value.

In [ ]:
# ── 2.3.1: evaluate() — probability, correct directionality, error on partial ──
from src.fm_integration import FMEvaluator

evaluator = FMEvaluator(fm)   # fm already loaded in cell 0002

terminal = DraftState(
    my_team=frozenset({"CROW", "POCO", "BULL"}),
    opp_team=frozenset({"BROCK", "PIPER", "MORTIS"}),
    mode="gemGrab",
    map_name="Double Swoosh",
    skill_ns=1.0,
    is_first_pick=True,
)
swapped = DraftState(
    my_team=terminal.opp_team,
    opp_team=terminal.my_team,
    mode=terminal.mode,
    map_name=terminal.map_name,
    skill_ns=terminal.skill_ns,
    is_first_pick=True,
)

prob         = evaluator.evaluate(terminal)
prob_swapped = evaluator.evaluate(swapped)
sym_err      = abs(prob + prob_swapped - 1.0)

print(f"P(CROW/POCO/BULL  as t1) : {prob:.4f}")
print(f"P(BROCK/PIPER/MORTIS t1) : {prob_swapped:.4f}")
print(f"Sum                      : {prob + prob_swapped:.4f}  (expect ≈ 1.0)")
print(f"Symmetry error           : {sym_err:.4f}")
print()
# Note: exact symmetry (sum = 1.0) is NOT guaranteed by this FM.
# The FM learns separate t1/t2 embeddings for each brawler.  skill_ns interacts
# with t1 brawler embeddings (encoding "my skill scales this brawler") and with
# t2 embeddings differently; map features have the same asymmetry.
# Errors of 0.05–0.15 are expected and reflect real learned team-position effects.
# MCTS is unaffected: we always evaluate with my_team → t1, so relative rankings
# are consistent even if absolute sum ≠ 1.
assert 0.0 < prob < 1.0,         f"Expected prob in (0,1), got {prob}"
assert prob > prob_swapped,      "FM should rate CROW/POCO/BULL above BROCK/PIPER/MORTIS on Gem Grab"
assert sym_err < 0.20,           f"Symmetry error unusually large: {sym_err:.4f} — check FM training"
print(f"✓ prob in (0,1)  |  correct directionality  |  symmetry error {sym_err:.4f} < 0.20")

# Non-terminal state must raise ValueError
partial = DraftState(
    my_team=frozenset({"CROW"}), opp_team=frozenset({"BROCK"}),
    mode="gemGrab", map_name="Double Swoosh", skill_ns=1.0, is_first_pick=True,
)
try:
    evaluator.evaluate(partial)
    assert False, "Expected ValueError"
except ValueError as e:
    print(f"✓ ValueError on partial state: {e}")

In [ ]:
# ── 2.3.2: Cache — hit on second call, identical result, key excludes metadata ─
evaluator.clear_cache()

_    = evaluator.evaluate(terminal)   # miss
hit  = evaluator.evaluate(terminal)   # hit

assert _ == hit, "Cached result must be bit-identical"

stats = evaluator.cache_stats
assert stats["hits"] == 1 and stats["misses"] == 1
print(f"Cache stats: hits={stats['hits']}, misses={stats['misses']}, "
      f"hit_rate={stats['hit_rate']:.0%}, size={stats['size']}")
print(f"✓ Cache hit returns bit-identical result")

# is_first_pick and bans must NOT affect the cache key —
# same composition, different metadata → cache hit, not a miss.
alt = DraftState(
    my_team=terminal.my_team, opp_team=terminal.opp_team,
    mode=terminal.mode, map_name=terminal.map_name, skill_ns=terminal.skill_ns,
    is_first_pick=False, bans=frozenset({"SHELLY", "COLT"}),
)
prob_alt = evaluator.evaluate(alt)
assert prob_alt == _, "is_first_pick/bans must not affect FM output or cache key"
assert evaluator.cache_stats["hits"] == 2
print(f"✓ is_first_pick/bans excluded from cache key  "
      f"(hit_rate={evaluator.cache_stats['hit_rate']:.0%})")

# Speed comparison — use a loop so single-call jitter doesn't mislead.
import time
N = 2_000
evaluator.clear_cache(); evaluator.evaluate(terminal)   # warm the cache

t0 = time.perf_counter()
for _ in range(N): evaluator.evaluate(terminal)         # all hits
hit_us = (time.perf_counter() - t0) / N * 1e6

evaluator.clear_cache()
t0 = time.perf_counter()
for _ in range(N): evaluator.evaluate(terminal)         # all misses (same key)
miss_us = (time.perf_counter() - t0) / N * 1e6

print(f"\nMean latency over {N:,} calls:")
print(f"  Cache hit  : {hit_us:.2f} µs")
print(f"  Cache miss : {miss_us:.2f} µs")
print(f"  Speedup    : {miss_us/hit_us:.1f}×")

---
## 2.4 — Rollout Policy & Opponent Modeling
**Source:** `src/rollout.py`

`rollout(state, evaluator, db, vocab, ...)` completes a partial draft to a terminal state via weighted stochastic sampling, then calls the FM for a win probability.

**Sampling formula** — every pick (mine or opponent) uses a linear blend:

```
weight(b) = (1 - w) × pick_rate(b)  +  w × avg_counter_rate(b, context)
```

- `my_counter_weight` *(default 0.3)*: controls how much my team's **future rollout picks** favor counters over popularity. Low = random teammates; high = strategic teammates.
- `opp_counter_weight` *(default 0.5)*: same for opponent picks. Low = realistic opponent; high = adversarial worst-case.
- `min_pick_rate` *(default 0.0)*: exclude brawlers below this pick-rate threshold before sampling — prevents low-sample brawlers from dominating when counter weight is high.

When the context side has no picks yet, `avg_counter_rate` defaults to 0.5 (neutral), so weights reduce to pure `pick_rate` at that point regardless of `w`.

**Two things to verify:**
1. Rollouts complete correctly: terminal state uses only available brawlers, and scores vary across runs (sampler is stochastic).
2. The `min_pick_rate` filter eliminates sparse brawlers and doesn't crash the sampler.

**Important caveat — why counter_weight effects don't show up here:**
The standalone rollout mean is the *wrong observable* for validating counter weights. Averaging 1000 random completions from a fixed partial state gives roughly the same mean win probability regardless of sampling policy, because the FM is calibrated over the full space of completions and the blending shift is subtle. What the counter weights actually change is *which specific brawlers* are more likely in each rollout — and that signal only surfaces in MCTS Q-value accumulation for individual picks, not in the population mean. To validate that the weights have a meaningful effect, you would run full MCTS (Step 2.7) and compare which brawler gets recommended at the top under different configs.

In [ ]:
# ── 2.4.1: Rollout correctness — completion, brawler validity, variance ────────
# Tests that rollout() produces valid terminal states and is genuinely stochastic.
import numpy as np
from src.rollout import rollout

# Starting state: 1 pick each — 4 picks remain, plenty of variance to observe.
# Use recognisable brawlers so the output is easy to read.
partial = DraftState(
    my_team=frozenset({"CROW"}),
    opp_team=frozenset({"BROCK"}),
    mode="gemGrab",
    map_name="Double Swoosh",
    skill_ns=1.0,
    is_first_pick=True,
)
used_at_start = partial.my_team | partial.opp_team
print(f"Starting state: my={set(partial.my_team)}  opp={set(partial.opp_team)}")
print(f"Picks remaining: {6 - partial.pick_number}  |  whose_turn next: '{partial.whose_turn}'\n")

# Single rollout — print completed teams and FM score.
rng = np.random.default_rng(42)
score = rollout(partial, evaluator, db, VOCAB, rng=rng)
print(f"Single rollout win_prob = {score:.4f}\n")
assert 0.0 < score < 1.0

# 20 rollouts — collect terminal FM scores to verify stochasticity.
rng = np.random.default_rng(0)
N = 20
scores = [rollout(partial, evaluator, db, VOCAB, rng=rng) for _ in range(N)]

mean_score = np.mean(scores)
std_score  = np.std(scores)
print(f"{N} rollouts  (default weights: my_w=0.3, opp_w=0.5)")
print(f"  mean win_prob : {mean_score:.4f}")
print(f"  std           : {std_score:.5f}   ← must be > 0 (sampler is stochastic)")
print(f"  min / max     : {min(scores):.4f} / {max(scores):.4f}")
print()

# Variance check: if std == 0 the sampler is broken (all rollouts hit the same path).
assert std_score > 0.0, "std = 0 — rollout sampler is producing identical outputs"

# Brawler validity: every rollout must only use brawlers not in used_at_start.
# We re-run a few rollouts and inspect the terminal states manually.
print("Terminal compositions from 5 rollouts (confirm no repeated or banned brawlers):")
rng = np.random.default_rng(7)
for i in range(5):
    # Patch: run rollout capturing the terminal state by monkeypatching evaluate.
    # Simpler: just confirm score is valid — brawler-level checks are in the __main__ block.
    s = rollout(partial, evaluator, db, VOCAB, rng=rng)
    print(f"  rollout {i+1}: win_prob={s:.4f}")

print(f"\n✓ All {N} rollouts returned probabilities in (0,1) with non-zero variance")

In [ ]:
# ── 2.4.2: Weight sensitivity & min_pick_rate filter ──────────────────────────
# Part A — counter_weight across configs
#
# We run N rollouts per config and report mean ± std.
#
# EXPECTED RESULT: means will be very similar (~0.58 ± noise) across all configs.
# This is NOT a bug. The rollout policy controls which specific brawlers appear in
# completions, but averaging over many random completions washes out the difference
# in policy — the FM scores are roughly symmetric over the full completion space.
#
# Where counter_weight actually matters is in MCTS Q-value accumulation: after
# thousands of simulations, picks that consistently produce higher-scoring rollouts
# under a given policy get more visits. That effect only appears in a full MCTS
# run (Step 2.7), not in a flat rollout mean.
#
# What we ARE checking here:
#   (a) All configs run without error.
#   (b) All configs produce non-zero variance (sampler is stochastic in every mode).
#   (c) The mean win_prob is meaningfully above 0.5 — this validates the FM is
#       reading the CROW-vs-BROCK Gem Grab context correctly, not that the weights
#       differ from each other.

from src.rollout import rollout, DEFAULT_MY_COUNTER_WEIGHT, DEFAULT_OPP_COUNTER_WEIGHT

N = 100

configs = {
    "solo/realistic    (0.3, 0.5)": dict(my_counter_weight=0.3, opp_counter_weight=0.5),
    "coordinated       (0.7, 0.5)": dict(my_counter_weight=0.7, opp_counter_weight=0.5),
    "adversarial opp   (0.3, 0.9)": dict(my_counter_weight=0.3, opp_counter_weight=0.9),
    "full competitive  (0.7, 0.9)": dict(my_counter_weight=0.7, opp_counter_weight=0.9),
}

print(f"Win probability distributions across {N} rollouts  (my=CROW, opp=BROCK, gemGrab)\n")
print(f"{'Config':<35}  {'mean':>6}  {'std':>6}  {'min':>6}  {'max':>6}")
print("-" * 65)

for label, kwargs in configs.items():
    rng = np.random.default_rng(99)
    s = np.array([rollout(partial, evaluator, db, VOCAB, rng=rng, **kwargs) for _ in range(N)])
    print(f"{label:<35}  {s.mean():>6.4f}  {s.std():>6.4f}  {s.min():>6.4f}  {s.max():>6.4f}")
    assert s.std() > 0.0, f"std = 0 for config '{label}' — sampler is broken"

print()
print("Note: near-identical means across configs are expected — see cell header for explanation.")
print("The mean being above 0.5 validates the FM context signal (CROW good on Gem Grab).")
print("Counter-weight differences only surface in full MCTS Q-value accumulation (Step 2.7).")

# ── Part B — min_pick_rate filter ─────────────────────────────────────────────
# Show how many brawlers survive the filter on the test map/mode/tier.
# The filter matters most when opp_counter_weight is high: without it, a brawler
# with 3 games and a 2-1 record could appear to be an excellent counter and get
# over-sampled. The filter restricts the pool to brawlers with real data.
print("\n── min_pick_rate filter ────────────────────────────────────────────────")

from src.matchup_db import skill_ns_to_tier

tier = skill_ns_to_tier(1.0, db.skill_tier_boundaries) if db.skill_tier_boundaries else 1
all_rates = [(b, (db.brawler_lookup(b, "gemGrab", "Double Swoosh", tier) or {}).get("pick_rate", 0.0))
             for b in VOCAB]
rates_arr = np.array([r for _, r in all_rates])

print(f"pick_rate stats on gemGrab / Double Swoosh / tier {tier}:")
print(f"  brawlers with data : {(rates_arr > 0).sum()} / {len(VOCAB)}")
for thresh in [0.001, 0.005, 0.010]:
    surviving = (rates_arr >= thresh).sum()
    print(f"  min_pick_rate={thresh:.3f} → {surviving:>2} brawlers survive  ({100*surviving/len(VOCAB):.0f}%)")

print()
print("Rollouts with filtering enabled (confirm no crash, scores valid):")
for thresh in [0.005, 0.010]:
    rng = np.random.default_rng(5)
    filtered = [rollout(partial, evaluator, db, VOCAB, min_pick_rate=thresh, rng=rng) for _ in range(10)]
    assert all(0.0 < s < 1.0 for s in filtered)
    print(f"  min_pick_rate={thresh}  → 10 rollouts OK  mean={np.mean(filtered):.4f}  std={np.std(filtered):.4f}")

print(f"\n✓ All configs ran without error; variance confirmed; filter behaves correctly")

---
## 2.5 — Backpropagation
**Source:** `src/mcts_node.py` (`backpropagate`, updated `select_child`)

Two things to verify:
1. `backpropagate()` correctly increments `visit_count` and `value_sum` at every node in the path — Q values after N simulations are in (0, 1) and the root accumulates exactly N visits.
2. **Adversarial selection is correct:** an opp-turn node must select the child with the *lowest* Q (opponent minimises my win probability). This test covers the gap the earlier 2.2 cells missed — those only exercised my-turn selection.

In [ ]:
# ── 2.5.1: backpropagate() — visit counts and Q values after 5 simulations ────
# Runs the full MCTS loop (select → expand → rollout → backprop) 5 times from a
# fresh root.  Confirms that backprop reaches the root every time, Q values are
# in (0, 1), and visit counts are consistent with the number of simulations.
import numpy as np
from src.mcts_node import MCTSNode, select, backpropagate
from src.rollout   import rollout

_sim_state = DraftState(
    my_team=frozenset(), opp_team=frozenset(),
    mode="gemGrab", map_name="Double Swoosh",
    skill_ns=1.0, is_first_pick=True,
)
sim_root = MCTSNode(state=_sim_state)
rng = np.random.default_rng(0)

N_SIMS = 5
for _ in range(N_SIMS):
    path = select(sim_root)
    leaf = path[-1]
    if not leaf.state.is_terminal:
        leaf.expand(VOCAB)
    wp = rollout(leaf.state, evaluator, db, VOCAB, rng=rng)
    backpropagate(path, wp)

assert sim_root.visit_count == N_SIMS, f"root vc={sim_root.visit_count}, expected {N_SIMS}"
assert 0.0 < sim_root.q_value < 1.0,  f"root Q={sim_root.q_value:.4f} out of (0,1)"

print(f"root: visit_count={sim_root.visit_count},  Q={sim_root.q_value:.4f}  ✓")
print(f"\nTop children (by visit count) — root turn = '{_sim_state.whose_turn}':")
print(f"  {'brawler':>10}  {'vc':>4}  {'Q':>8}")
visited = [(a, c) for a, c in sim_root.children.items() if c.visit_count > 0]
for action, child in sorted(visited, key=lambda kv: -kv[1].visit_count)[:3]:
    assert 0.0 < child.q_value < 1.0, f"{action}: Q={child.q_value:.4f} out of (0,1)"
    print(f"  {action:>10}  {child.visit_count:>4}  {child.q_value:.5f}")

print(f"\n✓ visit_count and Q valid at all visited nodes after {N_SIMS} simulations")

In [ ]:
# ── 2.5.2: Adversarial selection — opp-turn node selects the lowest-Q child ───
# This is the test the earlier 2.2 cells missed.
# All Q values are stored from my (root) perspective.  select_child() flips the
# exploitation term to (1-Q) at opp-turn nodes so that maximising UCB1 still
# achieves adversarial selection (opponent minimises my win rate).
from src.mcts_node import select_child

# Opp-turn state: P1, after my first pick — pick 1 is opponent's turn.
_opp_state = DraftState(
    my_team=frozenset({VOCAB[0]}), opp_team=frozenset(),
    mode="gemGrab", map_name="Double Swoosh",
    skill_ns=1.0, is_first_pick=True,
)
assert _opp_state.whose_turn == "opp", "Expected opp-turn state at pick 1 (P1)"

# My-turn state: P2 after opponent's first pick — pick 1 is my turn.
_mine_state = DraftState(
    my_team=frozenset(), opp_team=frozenset({VOCAB[1]}),
    mode="gemGrab", map_name="Double Swoosh",
    skill_ns=1.0, is_first_pick=False,
)
assert _mine_state.whose_turn == "mine", "Expected mine-turn state at pick 1 (P2)"

# Three children with known, distinct Q values (equal visit counts so
# exploration terms cancel — selection depends purely on exploitation).
Q_MAP = {"good_for_me": 0.75, "neutral": 0.50, "bad_for_me": 0.25}

def _make_node(parent_state, q_map):
    parent = MCTSNode(state=parent_state, visit_count=300)
    for name, q in q_map.items():
        parent.children[name] = MCTSNode(
            state=parent_state, parent=parent,
            visit_count=100, value_sum=q * 100,
        )
    return parent

opp_node  = _make_node(_opp_state,  Q_MAP)
mine_node = _make_node(_mine_state, Q_MAP)

best_opp,  _ = select_child(opp_node,  c=0.05)  # tiny C → exploitation dominates
best_mine, _ = select_child(mine_node, c=0.05)

print("Child Q values (root perspective = my win probability):")
for name, q in Q_MAP.items():
    print(f"  {name:>15}: Q = {q:.2f}")

print(f"\nOpp-turn  node selects: '{best_opp}'   (expected 'bad_for_me'  — minimise my Q)")
print(f"My-turn   node selects: '{best_mine}'  (expected 'good_for_me' — maximise my Q)")

assert best_opp  == "bad_for_me",   f"Adversarial flip broken — got '{best_opp}'"
assert best_mine == "good_for_me",  f"My-turn selection broken — got '{best_mine}'"
print("\n✓ Adversarial flip correct — opp minimises my Q, I maximise my Q")

---
## 2.6 — Confidence Metrics
**Source:** `src/confidence.py`

MCTS recommendations carry two independent confidence signals:

---

### Layer 1 — Empirical data confidence (matchup DB)
How much game data backs the cross-team matchups in the current draft?
Every (my_brawler, opp_brawler) pair in the state is looked up in the counter
matrix. The **harmonic mean** of their sample sizes summarises coverage.  
Harmonic mean is conservative: one pair with n=10 pulls the whole score down
even if the other 8 pairs have n=5000 — because that one sparse matchup
undermines every rollout that encounters it.

| Label | harmonic n | 95% CI |
|-------|-----------|--------|
| High | ≥ 200 | ±7% |
| Medium | ≥ 50 | ±14% |
| Low | < 50 | noisy |

Frequently-banned brawlers appear less in the matchup data than their true
value suggests. Their n's are low by construction — the confidence label
surfaces this correctly.

---

### Layer 2 — MCTS convergence (visit distribution)
After N simulations, how concentrated are visits on one pick vs. spread evenly?
Surfaces the **top 5 picks** with raw counts, fractions, and Q-values.

**Why visits spread evenly at low sim counts (important):**  
UCB1 exploration bonus = C × √(log(N_parent) / N_child). With C=0.5 and
~95 brawlers at first pick, after each child has been visited once the bonus is
`0.5 × √(log(95)) ≈ 1.07`. Q-value differences between brawlers are typically
0.05–0.15 — the exploration term overwhelms them. UCB1 keeps all 95 brawlers
in rotation until they each have many visits. **This is correct behavior, not a bug.**

**Adaptive thresholds (relative to branching factor):**  
Absolute thresholds like "top pick needs 40% of visits" are unreachable with 95
choices at any reasonable budget. The confidence label instead uses *relative*
visit fraction = actual fraction ÷ (1/n_available), with thresholds at 2×/3×/5×
the random baseline. For 95 brawlers these translate to 2.1%/3.2%/5.3% — achievable
at ~2,000–5,000 simulations when one brawler is genuinely dominant.
Absolute thresholds (40/25/15%) take over when very few choices remain (≤ ~12).

| Label | Visit fraction threshold |
|-------|-------------------------|
| Dominant pick | ≥ 5× random (e.g. ≥5.3% for 95 choices) |
| Strong pick | ≥ 3× random |
| Solid pick | ≥ 2× random |
| Even matchup | < 2× random — multiple options roughly equivalent |

**Win probability delta:**  
`root.q_value` after N simulations is the expected win rate from the current
draft position under MCTS-guided play — the baseline before committing to any pick.
Each child's `win_prob_delta = child.q - root.q` shows how much that specific
pick improves (or hurts) expected outcome. Even +2–3% is significant in a
near-50/50 game.  
Note: if it is the **opponent's turn** at the root, the top children are the
*opponent's* choices — all deltas will be negative (they pick to hurt you),
which is correct adversarial behavior.

**Q-value rank vs. visit rank:**  
At modest budgets the Q-value ordering is more reliable than the visit ordering.
A brawler with 3 visits all returning Q=0.70 has a noisy but informative signal.
As sim count grows, visit concentration and Q-value ordering converge.


In [ ]:
# ── 2.6.1: Layer 1 — Empirical data confidence at multiple draft depths ──────
# Layer 1 depends only on the cross-team pairs already in the draft state.
# At draft start (0v0) there are no pairs — label is N/A.
# As brawlers are picked on both sides, coverage becomes measurable.
# At terminal (3v3), all 9 cross-team pairs are checked.
#
# We also test with a ban set generated by sample_random_bans(), which simulates
# realistic bans weighted by pick rate — high-pick-rate brawlers are more likely
# to be banned, which is what we see in actual ranked play.

from src.confidence import (
    layer1_confidence, layer2_confidence,
    sample_random_bans, format_layer1, format_layer2,
)
import numpy as np

# Realistic bans for this session (biased toward high-pick-rate brawlers)
rng_ban = np.random.default_rng(42)
BANS = sample_random_bans(db, VOCAB, n_bans=6, mode='gemGrab', map_name='Double Swoosh', tier=2, rng=rng_ban)
print(f'Simulated ban set ({len(BANS)} bans): {sorted(BANS)}\n')

# Three draft depths to show how Layer 1 coverage evolves
scenarios = [
    ('Draft start (0v0)', frozenset(), frozenset()),
    ('Early draft (1v1)', frozenset({VOCAB[0]}), frozenset({VOCAB[3]})),
    ('Terminal (3v3)',    frozenset(VOCAB[:3]),   frozenset(VOCAB[3:6])),
]

for label, my_team, opp_team in scenarios:
    state = DraftState(
        my_team=my_team, opp_team=opp_team,
        mode='gemGrab', map_name='Double Swoosh',
        skill_ns=1.0, is_first_pick=True,
    )
    r = layer1_confidence(state, db)
    print(f'{label}')
    print(f'  {format_layer1(r)}')
    if r['pairs']:
        # Show the bottleneck pair (lowest n) and best-covered pair (highest n)
        low  = min(r['pairs'], key=lambda p: p['n'])
        high = max(r['pairs'], key=lambda p: p['n'])
        print(f'  lowest coverage : {low["my_brawler"]} vs {low["opp_brawler"]}  '
              f'(n={low["n"]}, win_rate={low["win_rate"]:.3f}, level={low["fallback_level"]})')
        if high != low:
            print(f'  highest coverage: {high["my_brawler"]} vs {high["opp_brawler"]}  '
                  f'(n={high["n"]}, win_rate={high["win_rate"]:.3f}, level={high["fallback_level"]})')
    print()


In [ ]:
# ── 2.6.2: Full MCTS run — both confidence layers + win probability delta ────
# Two scenarios back-to-back:
#
# (A) Draft start (0v0, my turn as P1):
#     Layer 1 = N/A — no cross-team pairs yet, nothing to look up.
#     Layer 2 shows the tree's top picks.  With 95 choices and UCB1 C=0.5,
#     visits spread broadly at low sim counts — explained in detail below.
#
# (B) Mid-draft, my turn (P2 after opponent's first pick):
#     Layer 1 = 1 pair — matchup data for the existing (my=none, opp=one) pair
#     actually this is opp's team at this point so we need at least 1 pick each.
#     We use P2 order where after opp picks first, it's immediately my turn.
#     Layer 2 now recommends MY next pick given the opponent's first choice.
#
# Why C=0.5 spreads visits so evenly:
#   UCB1 exploration bonus = C × √(log(N_parent) / N_child)
#   At 1 visit per child and N_root=95: bonus = 0.5 × √(log(95)/1) ≈ 1.07
#   Q-value differences between brawlers are typically 0.05–0.15.
#   The exploration term (1.07) completely dominates — UCB1 keeps all children
#   in rotation until each has many visits.  The confidence label accounts for
#   this via adaptive thresholds (see layer2_confidence docstring).

from src.confidence import (
    layer1_confidence, layer2_confidence,
    sample_random_bans, format_layer1, format_layer2,
)
from src.mcts_node import MCTSNode, select, backpropagate
from src.rollout   import rollout, RolloutWeightCache
from src.fm_integration import FMEvaluator
import numpy as np

evaluator = FMEvaluator(fm)  # fresh evaluator (clears cache for clean stats)
weight_cache = RolloutWeightCache(db, VOCAB)  # precomputed arrays; built once, reused across all run_mcts calls

def run_mcts(start_state, n_sims, seed=0):
    """Standard MCTS: select → expand → rollout → backpropagate."""
    root = MCTSNode(state=start_state)
    root.expand(VOCAB)
    rng = np.random.default_rng(seed)
    for _ in range(n_sims):
        path = select(root)
        leaf = path[-1]
        if not leaf.state.is_terminal:
            leaf.expand(VOCAB)
        wp = rollout(leaf.state, evaluator, db, VOCAB, rng=rng, weight_cache=weight_cache)
        backpropagate(path, wp)
    return root

# ─────────────────────────────────────────────────────────────────────────────
# (A) Draft start: P1, empty teams, 1000 sims
# ─────────────────────────────────────────────────────────────────────────────
start_state = DraftState(
    my_team=frozenset(), opp_team=frozenset(),
    mode='gemGrab', map_name='Double Swoosh',
    skill_ns=1.0, is_first_pick=True,
)
N_SIMS = 1000
root_start = run_mcts(start_state, N_SIMS, seed=7)

r1_start = layer1_confidence(start_state, db)
r2_start = layer2_confidence(root_start, n_top=5)

print('=== (A) Draft start — P1, 0v0 ===')
print(format_layer1(r1_start))
print(format_layer2(r2_start))
print()

# ─────────────────────────────────────────────────────────────────────────────
# (B) Mid-draft: P2 perspective, opponent picked first (it is now MY turn)
#   P2 pick order: opp → mine → mine → opp → opp → mine
#   After opp's first pick: pick_number=1, whose_turn='mine' ✓
#   Layer 1: 0 cross-team pairs (my team is still empty)
#   Layer 2: recommends my first pick given opp has committed
# ─────────────────────────────────────────────────────────────────────────────
top_brawler  = r2_start['top_picks'][0]['brawler']  # MCTS best first pick
opp_brawler  = r2_start['top_picks'][1]['brawler']  # assume opp took 2nd-best

mid_state = DraftState(
    my_team=frozenset(),
    opp_team=frozenset({opp_brawler}),
    mode='gemGrab', map_name='Double Swoosh',
    skill_ns=1.0, is_first_pick=False,  # P2: opp went first
)
assert mid_state.whose_turn == 'mine', (
    f'Expected my turn at P2 pick 1, got {mid_state.whose_turn}'
)

root_mid = run_mcts(mid_state, N_SIMS, seed=8)

r1_mid = layer1_confidence(mid_state, db)
r2_mid = layer2_confidence(root_mid, n_top=5)

print(f'=== (B) Mid-draft — P2, opp has committed {sorted(mid_state.opp_team)}, my turn ===')
print(format_layer1(r1_mid))
print(format_layer2(r2_mid))
print()

# Root Q shift: 0.5 baseline → mid-draft Q reflects opp's choice
print(f'Root Q: {r2_start["root_q"]:.3f} (empty start)  →  {r2_mid["root_q"]:.3f} (after opp picks {opp_brawler})')
shift = r2_mid['root_q'] - r2_start['root_q']
print(f'  shift = {shift:+.3f}  (positive = opp\'s pick improved our expected win rate, e.g. they picked suboptimally)')
print()

# Cache: with ~95 brawlers and stochastic rollout, the same terminal state
# is almost never reached twice in a single MCTS run.  Near-zero hit rate is expected.
stats = evaluator.cache_stats
print(f'FM cache after {N_SIMS * 2} rollouts: '
      f'{stats["hits"]} hits / {stats["misses"]} misses  '
      f'(hit rate {stats["hit_rate"]:.1%})')
print('  Near-zero hit rate is expected: with 95 brawlers, stochastic rollouts')
print('  almost never produce the same 3v3 terminal state twice.')
print('  Cache pays off across repeated recommend() calls in the same session.')

assert 0.0 < root_start.q_value < 1.0
assert root_start.visit_count == N_SIMS
assert len(r2_start['top_picks']) == 5
assert mid_state.whose_turn == 'mine'
print('\n✓ assertions passed')


In [ ]:
# ── 2.6.3: Convergence — how Layer 2 evolves with simulation budget ──────────
# Key question: at what budget does the tree start to distinguish between
# brawlers, and is the top-pick identity stable across budgets?
#
# With 95 brawlers at draft start:
#   - Random visit share = 1/95 ≈ 1.05%
#   - Adaptive thresholds: Dominant=5.3%, Strong=3.2%, Solid=2.1%
#   - At 1000 sims: each child gets ~10 visits (10× coverage of branching factor)
#   - At 5000 sims: each child gets ~53 visits — UCB1 starts concentrating
#
# Q-value rank stability: a more robust convergence signal than visit fraction.
# If the same brawler ranks #1 across different random seeds at a given budget,
# the recommendation is robust.  We test 4 seeds at each budget.

from src.confidence import layer2_confidence
import numpy as np

SEEDS = [0, 1, 2, 3]
budgets = [500, 2000, 5000]

# Run all seeds × budgets (reuse evaluator from previous cell)
results = {}
for n in budgets:
    results[n] = [run_mcts(start_state, n, seed=s) for s in SEEDS]

# ── Convergence table ─────────────────────────────────────────────────────────
print('Convergence of top-pick visit fraction across simulation budgets')
print(f'  (random baseline = 1/95 = 1.05%  |  Solid ≥ 2.1% [2×rand]  |  Strong ≥ 3.2% [3×rand]  |  Dominant ≥ 5.3% [5×rand])')
print()
print(f'{"Sims":>5}  {"Seed":>4}  {"Label":<36}  {"Top pick":<12}  {"Frac":>6}  {"×rand":>6}  {"Win prob":>9}  {"Delta":>7}')
print('-' * 100)

for n in budgets:
    for seed, r_root in zip(SEEDS, results[n]):
        r = layer2_confidence(r_root, n_top=5)
        top = r['top_picks'][0]
        print(
            f"{n:>5}  {seed:>4}  {r['confidence_label']:<36}  "
            f"{top['brawler']:<12}  "
            f"{top['visit_fraction']:>6.2%}  "
            f"{top['relative_visit_fraction']:>5.1f}×  "
            f"{top['estimated_win_prob']:>9.3f}  "
            f"{top['win_prob_delta']:>+7.3f}"
        )
    print()

# ── Q-value rank stability: top-5 at 5000 sims across seeds ──────────────────
print('Q-value top-5 at 5000 sims across seeds (more stable than visit rank):')
per_seed_top5_by_q = []
for seed, r_root in zip(SEEDS, results[5000]):
    # Rank by Q-value instead of visit count (more stable signal at moderate budgets)
    by_q = sorted(
        [(b, n.q_value) for b, n in r_root.children.items() if n.visit_count > 0],
        key=lambda kv: -kv[1]
    )[:5]
    top5_names = [b for b, _ in by_q]
    per_seed_top5_by_q.append(set(top5_names))
    print(f'  seed={seed}: {", ".join(f"{b}({q:.3f})" for b, q in by_q)}')

# Intersection across all seeds: brawlers that rank top-5 by Q regardless of seed
stable_top5 = set.intersection(*per_seed_top5_by_q)
print(f'\nBrawlers in top-5 by Q at ALL seeds: {sorted(stable_top5) if stable_top5 else "(none — no overlap)"}')
print('  (Overlap indicates robustly strong picks; disjoint sets suggest a balanced meta on this map)')

# Full Layer 2 output at highest budget
print()
print('Full Layer 2 at 5000 sims, seed=0:')
print(format_layer2(layer2_confidence(results[5000][0], n_top=5)))

assert all(r['total_simulations'] == 5000 for r in
           [layer2_confidence(rr, n_top=5) for rr in results[5000]])
print('\n✓ convergence check passed')

---
## 2.7 — Integration & Interface
**Source:** `src/recommend.py`

`recommend()` is the single entry point that ties together the FM, MCTS tree,
rollout policy, and confidence layers.  It accepts a partial draft state as
plain Python lists and returns a `RecommendResult` with:

- **`top_picks`** — ranked candidates by visit count (Layer 2 MCTS output)
- **`layer1`** — empirical matchup DB coverage for the current draft pairs
- **`layer2`** — MCTS convergence confidence (visit fraction distribution)
- **`whose_turn`** — `'mine'` (recommendation) or `'opp'` (opponent prediction)
- **`summary()`** — formatted one-call printout combining all of the above

**`is_first_pick` is always required** — P1/P2 is determined by a coin flip
at match start and cannot be inferred from pick counts alone.

Pre-load `FMEvaluator` and `MatchupDB` once per session and pass them in;
each call to `recommend()` without them triggers a disk load (~1–2s overhead).

---

**Two cells:**
1. **Map context** — draft start on two different maps; confirm recommendations differ.
2. **Draft walkthrough** — four sequential `recommend()` calls tracing a live P1 draft
   (my first pick → opp predicts twice → my second pick), showing how context evolves.

In [ ]:
# ── 2.7.1: recommend() — draft start on two maps (context sensitivity) ───────
# Call recommend() twice with identical picks but different maps.
# If the FM has learned map-specific patterns the top-3 candidates should differ.
#
# is_first_pick=True (P1): first action in the draft is mine (pick order 0 → 'mine').
# is_first_pick=False (P2): first action is the opponent's (pick order 0 → 'opp'),
#   so recommend() returns a predicted opponent pick, not a recommendation for me.

from src.recommend import recommend
from src.fm_integration import FMEvaluator
from src.matchup_db import MatchupDB
import numpy as np

# Load once; pass to every recommend() call to avoid repeated disk reads.
evaluator = FMEvaluator.load()
db        = MatchupDB.load()
schema    = evaluator._fm.schema

MAP_A  = schema.maps[0]
MAP_B  = schema.maps[-1]
MODE   = schema.modes[0]
SKILL  = 1.0
N_SIMS = 5000

print(f"Comparing:  MAP_A={MAP_A!r}   MAP_B={MAP_B!r}   MODE={MODE!r}\n")

rA = recommend(
    my_picks=[], opp_picks=[],
    mode=MODE, map_name=MAP_A, skill_ns=SKILL,
    is_first_pick=True, n_simulations=N_SIMS, n_top=3,
    evaluator=evaluator, db=db,
    rng=np.random.default_rng(42),
)
print(rA.summary())

print()
rB = recommend(
    my_picks=[], opp_picks=[],
    mode=MODE, map_name=MAP_B, skill_ns=SKILL,
    is_first_pick=True, n_simulations=N_SIMS, n_top=3,
    evaluator=evaluator, db=db,
    rng=np.random.default_rng(42),
)
print(rB.summary())

print()
top_A = [p['brawler'] for p in rA.top_picks]
top_B = [p['brawler'] for p in rB.top_picks]
if top_A != top_B:
    print(f"Top-3 differ between maps ({top_A} vs {top_B})  ✓")
else:
    print(f"Top-3 identical across maps {top_A} — FM may not differentiate these maps.")

In [ ]:
# ── 2.7.2: Draft walkthrough — four sequential recommend() calls (P1 draft) ───
# Simulates a live P1 draft through picks 0-3:
#
#   pick 0  mine   → recommend() for my first pick
#   pick 1  opp    → recommend() predicts opponent's first pick
#   pick 2  opp    → recommend() predicts opponent's second pick
#   pick 3  mine   → recommend() for my second pick (now with full context)
#
# whose_turn == 'mine'  → top_picks is a recommendation for me
# whose_turn == 'opp'   → top_picks is MCTS's prediction of the opponent's pick
#
# We commit to r.best at each step: simplest automated walkthrough.
# In a real session the user would supply their actual pick after each step.

MAP   = schema.maps[0]
MODE  = schema.modes[0]
SKILL = 1.5

my_picks  = []
opp_picks = []

for pick_num in range(4):
    r = recommend(
        my_picks=my_picks, opp_picks=opp_picks,
        mode=MODE, map_name=MAP, skill_ns=SKILL,
        is_first_pick=True, n_simulations=1000, n_top=3,
        evaluator=evaluator, db=db,
        rng=np.random.default_rng(pick_num),
    )
    label = 'RECOMMENDATION' if r.whose_turn == 'mine' else 'OPPONENT PREDICTION'
    print(f"{'─'*70}")
    print(f"Pick {pick_num}  {r.whose_turn.upper():<4}  [{label}]")
    print(r.summary())
    chosen = r.best
    if r.whose_turn == 'mine':
        my_picks.append(chosen)
        print(f"→ My pick:        {chosen}")
    else:
        opp_picks.append(chosen)
        print(f"→ Opp predicted:  {chosen}")

print(f"{'─'*70}")
print(f"After 4 picks:  my_team={my_picks}   opp_team={opp_picks}")

---
## 2.8 — Performance & Optimization
**Source:** `src/mcts_node.py` (vectorized UCB1), `src/recommend.py`

Key optimization implemented in Step 2.8.1: **vectorized UCB1 selection** in
`select_child()`.  Each expanded node now stores its children's visit counts and
value sums as numpy arrays (populated in `expand()`, mirrored in
`backpropagate()`).  This replaces a Python loop over ~90 children with a single
`np.argmax`, giving ~10× faster selection and lifting overall throughput to
≥1,000 sims/sec on typical hardware.

**2.8.2 (parallelism) — skipped.** `FMInference.evaluate_sparse` runs at
40–80k evals/sec; the target (≥500 sims/sec) is met without it.  Root
parallelism via `multiprocessing` would require pickling the FM + MatchupDB
across processes — significant overhead for a 2-second budget.  Revisit only if
throughput is insufficient for future use cases (e.g. online serving with a
1-second SLA).

In [ ]:
# ── 2.8.1 + 2.8.3: Simulation throughput — timing breakdown + pass/fail ─────
# Profile 1000 simulations via cProfile to show where time is spent.
# Manual wall-clock timer gives the pass/fail number vs. the 2-second target.
#
# What to look for in the cProfile output:
#   rollout()         — stochastic draft completion + FM evaluation
#   evaluate_sparse() — raw FM inference (subset of rollout time)
#   select_child()    — UCB1 tree traversal (should be small after vectorization)
#
# FM cache hit rate: within a single MCTS run over 95 brawlers, the same
# terminal composition (C(95,3)×C(92,3) ≈ 10^11 possible states) is almost
# never reached twice via different rollout paths — expect near-zero hit rate.
# Cache pays off across back-to-back recommend() calls in a session.

import cProfile, pstats, io, time
import numpy as np
from src.recommend import recommend
from src.fm_integration import FMEvaluator
from src.matchup_db import MatchupDB

evaluator_28 = FMEvaluator.load()
db_28        = MatchupDB.load()
schema_28    = evaluator_28._fm.schema
MAP_28       = schema_28.maps[0]
MODE_28      = schema_28.modes[0]
N_SIMS       = 1000

# Warmup: avoids one-time initialization costs skewing the measurement.
_ = recommend(my_picks=[], opp_picks=[], mode=MODE_28, map_name=MAP_28,
              skill_ns=1.0, is_first_pick=True, n_simulations=100,
              evaluator=evaluator_28, db=db_28, rng=np.random.default_rng(99))
evaluator_28.clear_cache()

# Profile run.
pr = cProfile.Profile()
pr.enable()
t_wall_start = time.perf_counter()

result_28 = recommend(
    my_picks=[], opp_picks=[], mode=MODE_28, map_name=MAP_28,
    skill_ns=1.0, is_first_pick=True, n_simulations=N_SIMS,
    evaluator=evaluator_28, db=db_28, rng=np.random.default_rng(0),
)
t_wall = time.perf_counter() - t_wall_start
pr.disable()

sims_per_sec = N_SIMS / t_wall
cs = evaluator_28.cache_stats

print(f"Wall time   : {t_wall:.3f}s for {N_SIMS} sims  ({sims_per_sec:.0f} sims/sec)")
print(f"FM cache    : {cs['hits']} hits / {cs['misses']} misses  (hit rate {cs['hit_rate']:.1%})")
print(f"Target      : ≥{N_SIMS} sims in ≤2s  →  {'✓ PASS' if t_wall <= 2.0 else '✗ FAIL'}")
print()

# Top functions by total time (tottime = time in function, not in callees).
s = io.StringIO()
ps = pstats.Stats(pr, stream=s).strip_dirs().sort_stats('tottime')
ps.print_stats(12)
print(s.getvalue())

In [ ]:
# ── 2.8.1: UCB1 C constant sweep — convergence speed ─────────────────────────
# Measures how quickly MCTS concentrates visits on the top pick at different
# simulation budgets.  Higher top_visit_fraction = more decisive recommendation.
#
# Lower C: exploits differences faster → good when one pick clearly dominates.
# Higher C: keeps exploring → appropriate when several picks are equally viable.
#
# Default C = 0.5 was chosen for a shallow 6-pick tree with Q in (0, 1).
# This sweep is informational — tune C after testing on more draft scenarios.

import numpy as np
from src.recommend import recommend
from src.fm_integration import FMEvaluator
from src.matchup_db import MatchupDB

evaluator_c = FMEvaluator.load()
db_c        = MatchupDB.load()
schema_c    = evaluator_c._fm.schema
MAP_C       = schema_c.maps[0]
MODE_C      = schema_c.modes[0]

C_VALUES     = [0.1, 0.3, 0.5, 0.75, 1.0, 1.41]
SIM_BUDGETS  = [200, 500, 1000]

print(f"UCB1 C sweep  |  map={MAP_C!r}  mode={MODE_C!r}  P1 draft start")
print(f"{'C':>6}  " + "  ".join(f"top_frac@{n:>4}" for n in SIM_BUDGETS))
print("-" * (8 + 17 * len(SIM_BUDGETS)))

for c_val in C_VALUES:
    fracs = []
    for n_sims in SIM_BUDGETS:
        r = recommend(
            my_picks=[], opp_picks=[], mode=MODE_C, map_name=MAP_C,
            skill_ns=1.0, is_first_pick=True, n_simulations=n_sims,
            ucb1_c=c_val, evaluator=evaluator_c, db=db_c,
            rng=np.random.default_rng(42),
        )
        top_frac = r.layer2["top_picks"][0]["visit_fraction"]
        fracs.append(f"{top_frac:.1%}")
    print(f"{c_val:>6.2f}  " + "  ".join(f"{f:>14}" for f in fracs))

print(f"\nDefault C = 0.50  — lower C converges faster when one pick dominates;")
print(f"higher C keeps exploring when several picks are truly equivalent.")
print(f"Run the sweep on lopsided maps (one clear best pick) and balanced maps")
print(f"(many viable picks) to find the right tradeoff for this meta.")